# Day 89: Expert Parallelism Simulation

**Layer:** Advanced


## Concept Overview

Expert parallelism shards the expert pool across GPUs. Each GPU holds E/N experts. All-to-all communication dispatches tokens to the GPU holding the selected expert, then gathers outputs.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Expert Dispatch Simulation

Simulate all-to-all token routing in expert parallelism and measure communication volume.


In [ ]:
import numpy as np

def simulate_expert_parallel(n_tokens=4096, n_experts=8, n_gpus=8, top_k=2):
    """Simulate expert parallel routing and communication."""
    np.random.seed(42)
    experts_per_gpu = n_experts // n_gpus
    
    # Token routing: each token selects top_k experts
    selected = np.random.choice(n_experts, size=(n_tokens, top_k), replace=False).reshape(n_tokens, top_k)
    
    # Count tokens per GPU
    tokens_to_gpu = np.zeros(n_gpus)
    for tok_experts in selected:
        for e in tok_experts:
            gpu = e // experts_per_gpu
            tokens_to_gpu[gpu] += 1
    
    # Communication volume (FP16, d_model=4096)
    d_model = 4096
    bytes_per_token = d_model * 2  # FP16
    total_comm = tokens_to_gpu.sum() * bytes_per_token  # all-to-all send
    
    print(f"Tokens: {n_tokens}, Experts: {n_experts}, GPUs: {n_gpus}, top_k: {top_k}")
    print(f"Expected tokens/GPU: {n_tokens * top_k / n_gpus:.0f}")
    print(f"Actual tokens/GPU: {tokens_to_gpu.astype(int)}")
    print(f"Load imbalance: {tokens_to_gpu.std()/tokens_to_gpu.mean():.1%}")
    print(f"All-to-all comm volume: {total_comm/1e9:.2f} GB")
    print(f"NVLink BW (600 GB/s): {total_comm/600e9*1000:.1f}ms per all-to-all")

simulate_expert_parallel()


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- Expert parallelism shards the expert pool across GPUs.
- Expert parallelism shards the expert pool across GPUs. Each GPU holds E/N expert....
- Day 89 implementation complete.
